In [3]:
# Charlotte West — Me Search Data Analysis
# 01 — Data Pull
# Loads raw UCMR5 PFAS data, ZIP code data, and Census demographics

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import geopandas as gpd
import os

In [4]:
#have to slim down the ucmr dataset so that it is small enough to get onto github

ucmr_full = pd.read_csv(
    '/Users/charlottewest/Downloads/QSS20/Me Search Data/ucmr5-occurrence-data-by-state/UCMR5_All_MA_WY.txt',
    sep      = '\t',
    encoding = 'cp1252',
    dtype    = {'PWSID': str, 'ZIPCode': str}
)

print(f'Full file rows:    {len(ucmr_full)}')
print(f'Full file columns: {ucmr_full.columns.tolist()}')

#keeping only the relevant columns
cols_to_keep = [
    'PWSID',                   # join key to ZIP codes file
    'PWSName',                 # water system name
    'State',                   # for filtering to NJ
    'Size',                    # S or L (small/large system)
    'FacilityWaterType',       # surface water vs groundwater
    'Contaminant',             # PFAS type
    'AnalyticalResultValue',   # the actual concentration
    'AnalyticalResultsSign',   # < (non-detect) or = (detected)
    'MRL',                     # minimum reporting level
    'Units',                   # µg/L
    'CollectionDate',          # sample date
    'Region',                  # EPA region
]

ucmr_slim = ucmr_full[cols_to_keep].copy()

#Filter to NJ only
ucmr_nj = ucmr_slim[
    (ucmr_slim['State'] == 'NJ') &
    (ucmr_slim['PWSID'].str.startswith('NJ'))
].copy()

#Check size reduction
print(f'\nBefore: {len(ucmr_full)} rows, {len(ucmr_full.columns)} columns')
print(f'After:  {len(ucmr_nj)} rows, {len(ucmr_nj.columns)} columns')

Full file rows:    1150728
Full file columns: ['PWSID', 'PWSName', 'Size', 'FacilityID', 'FacilityName', 'FacilityWaterType', 'SamplePointID', 'SamplePointName', 'SamplePointType', 'AssociatedFacilityID', 'AssociatedSamplePointID', 'CollectionDate', 'SampleID', 'Contaminant', 'MRL', 'Units', 'MethodID', 'AnalyticalResultsSign', 'AnalyticalResultValue', 'SampleEventCode', 'MonitoringRequirement', 'Region', 'State', 'UCMR1SampleType']

Before: 1150728 rows, 24 columns
After:  56542 rows, 12 columns


In [6]:

#Load ZIP Code Data
nj_zc = pd.read_csv(
    '/Users/charlottewest/Downloads/QSS20/Me Search Data/ucmr5-occurrence-data-by-state/UCMR5_ZIPCodes.txt',
    sep="\t", encoding="cp1252",
    dtype={'PWSID': str, 'ZIPCODE': str}
)

In [7]:
#Load Census Data
census_df = pd.read_csv(
    '/Users/charlottewest/Downloads/QSS20/Me Search Data/DPO5 ACS/ACSDP5Y2024.DP05-Data.csv',
    header=0,
    skiprows=[1],
    dtype={'GEO_ID': str}
)

census_df

,GEO_ID,NAME,DP05_0001E,DP05_0001M,DP05_0002E,DP05_0002M,DP05_0003E,DP05_0003M,DP05_0004E,DP05_0004M,...,DP05_0104PM,DP05_0105PE,DP05_0105PM,DP05_0106PE,DP05_0106PM,DP05_0107PE,DP05_0107PM,DP05_0108PE,DP05_0108PM,Unnamed: 434
0,0400000US34,New Jersey,9343809,*****,4598393,534,4745416,534,96.9,0.1,...,0.1,(X),(X),6433714,(X),48.3,0.1,51.7,0.1,NaN
1,860Z200US07001,ZCTA5 07001,17121,1128,9860,747,7261,666,135.8,13.9,...,1.6,(X),(X),11991,(X),58.7,3.0,41.3,3.0,NaN
2,860Z200US07002,ZCTA5 07002,71553,58,35678,818,35875,816,99.5,4.5,...,0.8,(X),(X),47852,(X),48.5,1.4,51.5,1.4,NaN
3,860Z200US07003,ZCTA5 07003,53771,44,26660,617,27111,621,98.3,4.5,...,0.6,(X),(X),38709,(X),48.7,1.4,51.3,1.4,NaN
4,860Z200US07004,ZCTA5 07004,7824,317,3373,365,4451,339,75.8,12.6,...,0.9,(X),(X),5907,(X),44.3,5.0,55.7,5.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594,860Z200US08889,ZCTA5 08889,10193,627,5177,444,5016,381,103.2,10.9,...,0.8,(X),(X),8351,(X),49.1,2.8,50.9,2.8,NaN
595,860Z200US08890,ZCTA5 08890,17,4,17,4,0,14,-,**,...,84.6,(X),(X),17,(X),100.0,84.6,0.0,84.6,NaN
596,860Z200US08901,ZCTA5 08901,57263,69,30012,1179,27251,1171,110.1,9.0,...,0.4,(X),(X),30157,(X),50.8,2.8,49.2,2.8,NaN
597,860Z200US08902,ZCTA5 08902,43932,377,21558,657,22374,669,96.4,5.6,...,0.7,(X),(X),27252,(X),47.9,1.8,52.1,1.8,NaN


In [8]:
#Loading NJ district shapes from Census Tigres Files

import geopandas as gpd

nj_shapefile = gpd.read_file(
    "/Users/charlottewest/Downloads/QSS20/Me Search Data/tl_2024_us_zcta520"
)

#sorting it to relevant zipcodes
nj_shapefile = nj_shapefile[
    nj_shapefile['ZCTA5CE20'].str.startswith('07') |
    nj_shapefile['ZCTA5CE20'].str.startswith('08')
].copy()

# Rename to match your ZIPCODE column
nj_shapefile_slim = nj_shapefile.rename(columns={'ZCTA5CE20': 'ZIPCODE'})

nj_shapefile_slim.head()

,ZIPCODE,GEOID20,GEOIDFQ20,CLASSFP20,MTFCC20,FUNCSTAT20,ALAND20,AWATER20,INTPTLAT20,INTPTLON20,geometry
21637,07066,07066,860Z200US07066,B5,G6350,S,11337641,466241,+40.6206502,-074.3098621,"POLYGON ((-74.35868 40.60409, -74.35863 40.604..."
21638,07608,07608,860Z200US07608,B5,G6350,S,4431934,26306,+40.8496088,-074.0618930,"POLYGON ((-74.07352 40.84149, -74.07202 40.842..."
21639,08723,08723,860Z200US08723,B5,G6350,S,30976323,9466538,+40.0385836,-074.1116001,"POLYGON ((-74.16044 40.05879, -74.16037 40.059..."
21640,08088,08088,860Z200US08088,B5,G6350,S,389829618,4466960,+39.8373145,-074.6952670,"POLYGON ((-74.81451 39.78551, -74.81336 39.788..."
21641,08008,08008,860Z200US08008,B5,G6350,S,27995232,88485185,+39.6235376,-074.2060888,"POLYGON ((-74.3425 39.56545, -74.34226 39.5654..."


In [11]:

#Save outputs for next notebook
os.makedirs('data', exist_ok=True)

# Now save
ucmr_nj.to_csv('data/ucmr5_nj_slim.csv', index=False)
nj_zc.to_csv('data/zipcodes_raw.csv', index=False)
census_df.to_csv('data/census_raw.csv', index=False)

print("Data pull complete")
print(f"UCMR rows: {len(ucmr_nj)}")
print(f"ZIP rows: {len(nj_zc)}")
print(f"Census rows: {len(census_df)}")

Data pull complete
UCMR rows: 56542
ZIP rows: 31107
Census rows: 599


In [12]:
# Save NJ shapefile as a GeoJSON — single file

nj_shapefile_slim.to_file(
    'data/nj_zcta_2024.geojson',
    driver='GeoJSON'
)

In [39]:
for f in os.listdir('data'):
    size_mb = os.path.getsize(f'data/{f}') / (1024 * 1024)
    print(f"{f}: {size_mb:.1f} MB")

nj_zcta_2024.geojson: 14.7 MB
zipcodes_raw.csv: 0.5 MB
ucmr5_raw.csv: 175.5 MB
census_raw.csv: 1.0 MB
ucmr5_nj_slim.csv: 4.2 MB
